## Домашнее задание №4. Работа с NPV моделью

*Дата выдачи: 17.08.2026*

*Дедлайн: 24.08.2026 23:59*


### Задание 1: Написать расчет IRR
- Добавить метод для расчета IRR


### Задание 2: Исследовать NPV-модель на чувствительность 
- Меняем параметры вверх-вниз, считаем NPV для новых параметров
- Объясняем, как и почему изменился NPV


### Задание 3: добавить страховку в NPV модель
- Страховка стоит 0.5% от лимита
- Подключена у 40% клиентов в статусе CUR и у 60% в статусе DLQ
- Возможно наступление страхового случая с вероятностью 0.1%, при подключенной страховке мы должны возместить всю сумму текущей задолженности
- Нужно добавить страховку в модель и сделать выводы о том, как и почему меняется NPV

Вам нужно внести изменения в код ниже. Где – подсказывать не будем, это часть задания. Добавьте комментарий с указанием места, котороые вы поменяли.

### Класс модели

In [ ]:
import numpy as np
import scipy as sp 

class NPVModel:

    avg_missed_payments = 1.5
    recovery = 0.50
    dlq_penalty_amount = 500
    oper_costs = 100
    collection_costs = 600
    tax_rate = 0.25
    discounting_rate = 0.30
    eq_req = 0.125
    cost_of_funds = 0.16
    acquisition_cost = 1000

    # Здесь тоже меняем, чтобы
    # Сохранить инфу о
    # страховии из условия
    ins_cost = 0.005
    ins_rate_cur = 0.4
    ins_rate_dlq = 0.6
    ins_prob = 0.001

    def __init__(self):
        pass

    def model_balance_calculations(self, amount, rate, term):
        '''
        Расчет модельных баланса, выплаченных процентов, регулярного платежа
        :param amount: Сумма кредита
        :param rate: Ставка
        :param term: Срок
        :return:
        balance : np.array(101) : остаток тела долга по кредиту на каждый месяц
        interest : np.array(101) : выплата по процентам каждый месяц
        regular_payment : float : размер регулярного платежа
        '''

        regular_payment = round(amount * (rate / 12) *\
                                (1 + (rate / 12)) / (1 - ((1 + rate / 12) ** (-term))))
        # График баланса и процентов
        balance = np.zeros(101)
        interest = np.zeros(101)
        balance[0] = amount

        for i in range(1, term + 1):
            int_payment = balance[i - 1] * rate / 12
            debt_payment = regular_payment - int_payment
            balance[i] = max(0, round(balance[i - 1] - debt_payment))
            interest[i] = int_payment
        return balance, interest, regular_payment

    def distribution_calc(self, portfolio_distribution, pd=None, pa=None):
        '''
        Расчет распределения по статусам
        :param portfolio_distribution: график распределения по статусам
        :return:
        cur_dist : np.array(101) : доля клиентов в статусе CUR на каждый месяц
        dlq_dist : np.array(101) : доля клиентов в статусе DLQ на каждый месяц
        act_dist : np.array(101) : доля клиентов в статусе ACT на каждый месяц
        def_dist : np.array(101) : доля клиентов в статусе DEF на каждый месяц
        clo_dist : np.array(101) : доля клиентов в статусе CLO на каждый месяц
        '''
        
        cur_dist = portfolio_distribution['CUR']
        dlq_dist = portfolio_distribution['DLQ']
        act_dist = cur_dist + dlq_dist
        def_dist = portfolio_distribution['DEF'] 
        clo_dist = portfolio_distribution['CLO']


        return cur_dist, dlq_dist, act_dist, def_dist, clo_dist

    def cur_balance_calc(self, model_balance, cur_dist):
        '''
        Расчет модельного баланса в статусе CUR
        :param model_balance: плановый график баланса
        :param cur_dist: доля клиентов в статусе CUR на каждый месяц
        :return:
        principal_balance_cur : np.array(101) : principal balance в статусе CUR
        gross_balance_cur : np.array(101) : gross balance в статусе CUR
        '''
        principal_balance_cur = model_balance*cur_dist
        gross_balance_cur = model_balance*cur_dist
        return principal_balance_cur, gross_balance_cur

    def dlq_balance_calc(self, model_balance, regular_payment, dlq_dist):
        '''
        Расчет модельного баланса в статусе DLQ
        :param model_balance: плановый график баланса
        :param regular_payment: размер регулярного платежа
        :param dlq_dist: доля клиентов в статусе DLQ на каждый месяц
        :return:
        principal_balance_dlq : np.array(101) : principal balance в статусе DLQ
        gross_balance_dlq : np.array(101) : gross balance в статусе DLQ
        '''
        principal_balance_dlq = np.append(0, model_balance[:-1])*dlq_dist
        gross_balance_dlq = (model_balance + regular_payment*self.avg_missed_payments)*dlq_dist
        return principal_balance_dlq, gross_balance_dlq

    def act_balance_calc(self, principal_balance_cur, principal_balance_dlq, gross_balance_cur, gross_balance_dlq):
        '''
        Расчет модельного баланса в статусе ACT
        :param principal_balance_cur: principal balance в статусе CUR
        :param principal_balance_dlq: principal balance в статусе DLQ
        :param gross_balance_cur: principal balance в статусе CUR
        :param gross_balance_dlq: principal balance в статусе DLQ
        :param dlq_dist: доля клиентов в статусе DLQ на каждый месяц
        :return:
        principal_balance_act : np.array(101) : principal balance в статусе ACT
        gross_balance_act : np.array(101) : gross balance в статусе ACT
        '''
        principal_balance_act = principal_balance_cur + principal_balance_dlq
        gross_balance_act = gross_balance_cur + gross_balance_dlq
        return principal_balance_act, gross_balance_act

    def def_balance_calc(self, model_balance, regular_payment, def_dist):
        '''
        Расчет модельного баланса в статусе DEF
        :param model_balance: плановый график баланса
        :param regular_payment: размер регулярного платежа
        :param def_dist: доля клиентов в статусе DEF на каждый месяц
        :return:
        principal_balance_def : np.array(101) : principal balance в статусе DEF
        gross_balance_def : np.array(101) : gross balance в статусе DEF
        '''
        principal_balance_def = np.zeros(101)
        gross_balance_def = np.zeros(101)

        def_dist_change = def_dist[4:] - def_dist[3:-1]
        principal_balance_def[4:] = np.cumsum(model_balance[:-4] * def_dist_change)
        gross_balance_def[4:] = np.cumsum((model_balance[4:] + 4 * regular_payment) * def_dist_change)

        return principal_balance_def, gross_balance_def
    
    def profit_calc(self, principal_balance_act, principal_balance_def, term, rate, dlq_dist):
        '''

        :param principal_balance_act: principal balance в статусе ACT
        :param principal_balance_def: principal balance в статусе DEF
        :param term: срок кредита
        :param rate: тавка по кредиту
        :param dlq_dist: доля клиентов в статусе DLQ на каждый месяц
        :return:
        profit : np.array(101) : доход на каждый месяц
        '''
        interest_profit = principal_balance_act*rate/12
        penatly_profit = dlq_dist*self.dlq_penalty_amount

        new_def_balance = np.append(principal_balance_def[1:], 0) - principal_balance_def
        recovery_profit = new_def_balance*self.recovery

        profit = interest_profit + recovery_profit + penatly_profit
        profit[term+1:] = 0

        return profit

    def loss_calc(self, gross_balance_act, principal_balance_def, term, act_dist, dlq_dist, def_dist):
        '''

        :param gross_balance_act: gross balance в статусе ACT
        :param principal_balance_def: principal balance в статусе DEF
        :param term: срок кредита
        :param act_dist: доля клиентов в статусе ACT на каждый месяц
        :param dlq_dist: доля клиентов в статусе DLQ на каждый месяц
        :param def_dist: доля клиентов в статусе DEF на каждый месяц
        :return:
        loss : np.array(101) : лосс на каждый месяц
        '''
        oper_loss = np.append(act_dist[1:], 0)*self.oper_costs
        loan_loss =  principal_balance_def - np.append(0, principal_balance_def[:-1])
        collection_loss = (np.append(dlq_dist[1:], 0) + np.append(def_dist[1:], 0))*self.collection_costs

        prev_gross_balance_act = np.append(0, gross_balance_act[:-1])
        cost_of_funds_loss = prev_gross_balance_act*(1 - self.eq_req)*self.cost_of_funds/12

        loss = loan_loss + cost_of_funds_loss + oper_loss + collection_loss
        loss[term+1:] = 0

        return loss, loan_loss, cost_of_funds_loss, oper_loss, collection_loss

    def assets_liabilities_calc(self, gross_balance_act):
        '''

        :param gross_balance_act: gross balance в статусе ACT
        :return:
        assets : np.array(101) : активы в проекте
        eq_req_curve : np.array(101) : активы, обеспеченные капиталом
        fund_req_curve : np.array(101) : активы, обеспеченные фондами
        '''

        # assets
        assets = gross_balance_act

        #liabilities
        eq_req_curve = assets*self.eq_req
        fund_req_curve = assets*(1 - self.eq_req)

        return assets, eq_req_curve, fund_req_curve

    def niat_calc(self, profit, loss):
        '''
        Расчет NIAT
        :param profit: суммарный доход на каждый месяц
        :param loss: суммарный лосс на каждый месяц
        :return:
        nibt : np.array(101) : прибыль до налогообложения
        niat : np.array(101) : прибыль после налогообложения
        tax : np.array(101) : налог в каждом месяце
        '''

        nibt = profit - loss
        tax = nibt*self.tax_rate
        niat = nibt - tax

        return nibt, niat, tax

    def cashflow_calc(self, principal_balance_act, principal_balance_def, amount, profit,
                      cost_of_funds_loss, oper_costs, collection_costs, tax, niat, eq_req_curve, fund_req_curve):
        '''
        Расчет денежных потоков
        :param principal_balance_act: principal balance в статусе DEF
        :param principal_balance_def: principal balance в статусе DEF
        :param amount: срок кредита
        :param profit: доход на каждый месяц
        :param cost_of_funds_loss: лоссы на фондирование
        :param oper_costs: операционные расходы
        :param collection_costs: расходы на коллекшн
        :param tax: налог в каждом месяце
        :param niat: доход после налогообложения
        :param eq_req_curve: активы, обеспеченные капиталом
        :param fund_req_curve: активы, обеспеченные фондами
        :return:
        cf_to_client : np.array(101) : денежный поток к клиенту
        cf_to_shareholders : np.array(101) : денежный поток к акционерам
        cf_to_debtholders : np.array(101) : денежный поток к фондам
        cf_to_cost_and_tax : np.array(101) : денежный поток на косты и налоги
        '''
        # client
        delta_principal_balance_act = (np.append(0, principal_balance_act[:100]) - principal_balance_act)
        delta_principal_balance_def = (np.append(0, principal_balance_def[:100]) - principal_balance_def)
        repayments = delta_principal_balance_act - delta_principal_balance_def + profit
        cf_to_client = np.append(amount, -repayments[1:])

        # debtoholder
        fund_req_ch = fund_req_curve - np.append(fund_req_curve[0], fund_req_curve[:-1])
        fund_req_change = np.append(fund_req_curve[0], fund_req_ch[1:])
        cf_to_debtholders = cost_of_funds_loss - fund_req_change

        # shareholders
        eq_req_ch = eq_req_curve - np.append(eq_req_curve[0], eq_req_curve[:-1])
        cf_to_shareholders = niat - np.append(eq_req_curve[0], eq_req_ch[1:])

        # cost and tax
        cf_to_cost_and_tax = oper_costs + collection_costs + tax

        return cf_to_client, cf_to_shareholders, cf_to_debtholders, cf_to_cost_and_tax

    def npv_calc(self, amount, rate, term, portfolio_distribution, pd=None, pa=None, dr = None):
        '''
        Расчет NPV
        :param amount: сумма кредита
        :param rate: ставка по кредиту
        :param term: срок кредита
        :param portfolio_distribution: график распределения по статусам
        :param pd: вероятность дефолта к 12-ому месяцу
        :param pa: вероятность полного досрочного погашения к 6-ому месяцу
        :return: npv: чистая приведенная стоимость кредита
        '''

        # Считаем балансы
        model_balance, interest, regular_payment =  self.model_balance_calculations(amount, rate, term)

        # Считаем распределение
        cur_dist, dlq_dist, act_dist, def_dist, clo_dist = self.distribution_calc(portfolio_distribution)

        # Считаем балансы
        principal_balance_cur, gross_balance_cur = self.cur_balance_calc(model_balance, cur_dist)
        principal_balance_dlq, gross_balance_dlq = self.dlq_balance_calc(model_balance, regular_payment, dlq_dist)
        principal_balance_act, gross_balance_act = self.act_balance_calc(principal_balance_cur, principal_balance_dlq, gross_balance_cur, gross_balance_dlq)
        principal_balance_def, gross_balance_def = self.def_balance_calc(model_balance, regular_payment, def_dist)

        # Обновочка! Здесь мы
        # будем считать профит и
        # лоссы с кредитования.

        # Здесь смысл в том, что
        # у нас профит от страховки
        # идёт от лимита, поэтому
        # мы смотрим на изначальный amount.
        ins_profit_cur = amount * cur_dist * self.ins_rate_cur * self.ins_cost 
        ins_profit_dlq = amount * dlq_dist * self.ins_rate_dlq * self.ins_cost 

        # Здесь уже по условию 
        # мы возвращаем только
        # сумму текущей задолженности,
        # а это gross_balance.
        ins_loss_cur = gross_balance_cur * self.ins_rate_cur * self.ins_prob
        ins_loss_dlq = gross_balance_dlq * self.ins_rate_dlq * self.ins_prob

        # Конец обновочки.

        # Считаем профиты
        profit = self.profit_calc(principal_balance_act, principal_balance_def, term, rate, dlq_dist)
        loss, loan_loss, cost_of_funds_loss, oper_loss, collection_loss = self.loss_calc(gross_balance_act, principal_balance_def,
                                                                                                                  term, act_dist, dlq_dist, def_dist)
        # Тут тоже небольшие реновации, 
        # обновляем профиты и лоссы с
        # учётом страховки.
        profit += ins_profit_cur + ins_profit_dlq
        loss += ins_loss_cur + ins_loss_dlq

        nibt, niat, tax = self.niat_calc(profit, loss)


        # Считаем активы и денежные потоки
        assets, eq_req_curve, fund_req_curve = self.assets_liabilities_calc(gross_balance_act)
        cf_to_client, cf_to_shareholders, cf_to_debtholders, cf_to_cost_and_tax =\
            self.cashflow_calc(principal_balance_act, principal_balance_def, amount, profit, cost_of_funds_loss, oper_loss, collection_loss,
                               tax, niat, eq_req_curve, fund_req_curve)

        # Считаем NPV

        # Вот тут целый блок поменялся
        # начиная от нового dr
        # заканчивая новым pv

        if not dr and dr != 0: 
            discounting_rate = self.discounting_rate
        else: 
            discounting_rate = dr

        disc_curve = np.array([1 / ((1 + discounting_rate) ** (i / 12.)) for i in range(101)])
        # pv = np.round(np.sum(disc_curve * cf_to_shareholders)).astype(int)
        pv = np.sum(disc_curve * cf_to_shareholders)
        
        # конец изменений.

        npv = pv - self.acquisition_cost

        return npv


    def irr_calc(self, amount, rate, term, portfolio_distribution, pd=None, pa=None):

        def f(dr):
            return self.npv_calc(amount, rate, term, portfolio_distribution, pd, pa, dr = dr)
        return sp.optimize.root_scalar(f, x0 = 0.25, method = 'secant', bracket = [0., 2.5]).root

### Пример использования модели

In [110]:
cur_dist = np.zeros(101)
dlq_dist = np.zeros(101)
clo_dist = np.zeros(101)
def_dist = np.zeros(101)

dlq_dist[1:12] += 0.05

def_dist[6:] += 0.03
def_dist[9:] += 0.03

def_dist[12:] += dlq_dist[11]

clo_dist[3:] += 0.10
clo_dist[6:] += 0.10
clo_dist[12:] = 1 - def_dist[12]

cur_dist = 1 - def_dist - clo_dist - dlq_dist

portfolio_distribution = {
    'CUR' : cur_dist,
    'DLQ' : dlq_dist,
    'DEF' : def_dist,
    'CLO' : clo_dist
         }

In [111]:
import plotly.express as px
import pandas as pd

df = pd.DataFrame({
    'CUR': cur_dist,
    'DLQ': dlq_dist,
    'DEF': def_dist,
    'CLO': clo_dist
})

df = df.reset_index().rename(columns={'index': 'Номер выписки'})
df = df.melt(id_vars='Номер выписки', value_vars=['DEF', 'DLQ', 'CUR', 'CLO'], 
             var_name='Status', value_name='Доля')

fig = px.area(df, x='Номер выписки', y='Доля', color='Status', 
              title='Распределение статусов по номерам выписки',
              color_discrete_map={
                  'DEF': '#D62728',
                  'DLQ': '#FF7F0E',
                  'CUR': '#2CA02C',
                  'CLO': '#1F77B4',
              },
              category_orders={'Status': ['DEF', 'DLQ', 'CUR', 'CLO']}
             )

fig.update_layout(height=600)
fig.update_xaxes(range=[0, 15])

fig.show()

In [112]:
model = NPVModel()
model.npv_calc(amount=200000, term=12, rate=0.24, portfolio_distribution=portfolio_distribution), model.irr_calc(amount=200000, term=12, rate=0.24, portfolio_distribution=portfolio_distribution)

(np.float64(-1419.1774846634507), np.float64(0.10718311873132744))

### Задание 2.
Будем считать, что распределения никак не меняются и даны нам кем-то или чем-то с неоспоримым авторитетом, поэтому менять будем всё, кроме этого. 

In [138]:
res = []

for amount in [100_000 * i for i in range(1, 4)]:
    for term in [1, 2, 4, 6, 8, 12, 24, 36]:
        for rate in np.linspace(start = 0.05, stop = 0.80, num = 15):
            res.append((amount, term, rate, 
                        model.npv_calc(amount=amount, term=term, rate=rate, portfolio_distribution=portfolio_distribution)
            ))

len(res)

360

In [139]:
df = pd.DataFrame(res, columns = ['amount', 'term', 'rate', 'NPV'])
df.head()

,amount,term,rate,NPV
0,100000,1,0.050000,-2170.875447
1,100000,1,0.103571,-1821.437678
2,100000,1,0.157143,-1472.005791
3,100000,1,0.210714,-1122.583708
4,100000,1,0.264286,-773.167507


In [137]:
for amount in [100_000 * i for i in range(1, 4)]:
    fig = px.density_heatmap(
        df[df['amount'] == amount], 
        x='rate', 
        y='term', 
        z='NPV', 
        title=f'Amount: {amount}', 
        text_auto=True
    )
    fig.show()

Что отсюда очевидно: нам всегда невыгодно выдавать кредиты под маленькие ставки и на маленькие сроки, потому что тогда косты превысят проценты с клиента, и мы только потеряем от такого кредита, но это и вполне логично, так как такие сказочные условия могут быть только у дружеских заёмов. С другой стороны, если выдывать кредит на больший срок, то мы можем заработать чуть больше, но всё равно будем в диком минусе, потому что фактически ничего не зарабатываем, а из-за обесценивания денег мы постепенно получаем всё меньше и меньше денег, так ещё и люди могут в дефолт уйти. Такой же логичный вывод, что нам очень выгодно давать кредиты на короткий срок (до года) под дикие проценты, потому что мы за всё это время мы в сумме не так много денег потратим на операционные и прочие расходы, клиенты не успеют уйти в дефолт, а наши деньги не успеют обесцениться, поэтому мы крупно заработаем. Кроме того, заметим, что максимум NPV на всех графиках находится в условиях малых сроков, но при этом не максимальных процентах, то есть максимум NPV находится при $rate \in [0.4, 0.599]$, скорее всего это связано с формой платежей, где-то в этом отрезке находится оптимум для кредитной ставки, при которой мы получаем наибольшой процент, а дальше этого отрезка, соответсвенно, мы уже получаем меньше, значит, итог тоже меньше. Это касается абсолютно всех столбцов, при одинаковых сроках кредита нам всегда выгодно брать кредитную ставку из этого отрезка с оптимумом, что вполне естественно. В целом видно, что наша модель окупается во всех ситуациях, когда процентная ставка $\ge 0.2$, и если мы не играем в МФО и выдаём кредиты сроком от года, то нам выгоднее давать кредиты на больший срок, потому что мы так можем стабильно зарабатывать с процентов, а так как ставка достаточно большая, чтобы покрыть все расходы и обесценивание денег, то это получается довольно стабильный поток денег, основная проблема которого - дефолтники.

### Место для комментариев

Итак, так как я не понял, для каких комментариев именно это поле, то это будет место моих заметок и логгирования.
1) Здесь всё просто, нам нужен какой-нить солвер, который решит уравнение $NPV(r) := \sum_{i=1}^n \frac{CF_i}{(1 + r)^{\frac{i}{12}}} = 0$, для этого я чутка подкорректировал метод ```npv_calc```, чтобы он мог принимать ставку дисконтирования как аргумент, однако в самом методе есть небольшая ловушка с тем, что в итоге число округляется и приводится к инту, проблема с этим в том, что градиентные методы в итоге дают нулевой градиент, и мы никуда не сдвигаемся, а безградиентные просто никуда не сдвигаются, потому что не видят улучшение, поэтому теперь метод ничего не округляет и поэтому не является ступенчатой функцией, которая душит все методы.